# CSCI 6379 / 4353 &mdash; Topic 5
## Your First Model, End to End

Dr. Dongchul Kim &middot; Department of Computer Science, UTRGV &middot; Fall 2026

---

**You are not expected to understand the inside of any step here.** Today the goal is to see
the shape of the whole thing. Every box gets opened properly in a later topic.

The six boxes, always in this order:

`1 LOAD` &rarr; `2 SPLIT` &rarr; `3 MODEL` &rarr; `4 FIT` &rarr; `5 PREDICT` &rarr; `6 SCORE`

Run cells with **Shift + Enter**.


## Part 1. The whole program, in ten lines


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)                                  # 1 LOAD
X_train, X_test, y_train, y_test = train_test_split(               # 2 SPLIT
    X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=200)                           # 3 MODEL
model.fit(X_train, y_train)                                        # 4 FIT
pred = model.predict(X_test)                                       # 5 PREDICT
print('accuracy:', accuracy_score(y_test, pred))                   # 6 SCORE


A working classifier. Right on **29 of the 30 flowers it had never seen**.

Notice what is *not* there: no formulas, no gradients, no loop. The library holds the details.


## Part 2. The model really is empty before `fit`

Creating the model does no learning at all.


In [ ]:
empty = LogisticRegression(max_iter=200)
try:
    empty.predict(X[:3])
except Exception as e:
    print(type(e).__name__)
    print(e)


## Part 3. One number is not enough

Accuracy says there was one error. It does not say *what got confused with what*.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

iris = load_iris()
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, target_names=iris.target_names))


Rows are what the flower **actually was**, columns are what the model **said**.
The single `1` sits in the versicolor row, virginica column.

The same one error shows up twice: versicolor **recall** falls to 0.90 (a real one was missed)
and virginica **precision** falls to 0.91 (a false alarm). Column versus row, as in Topic 3.


---
# Part 4. The same six boxes in PyTorch

**This is the part that cannot run in the browser.** PyTorch adds no boxes; it makes you
write box 4 yourself.


In [ ]:
import torch, torch.nn as nn
print('torch:', torch.__version__)


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

torch.manual_seed(42)
model_t = nn.Linear(4, 3)                                   # 3 MODEL
loss_fn = nn.CrossEntropyLoss()                             # how wrong are we?
opt = torch.optim.SGD(model_t.parameters(), lr=0.1)         # how do we fix it?

for epoch in range(200):                                    # 4 FIT, by hand
    opt.zero_grad()                 # forget last round's corrections
    out = model_t(X_train_t)        # guess
    loss = loss_fn(out, y_train_t)  # measure how wrong
    loss.backward()                 # which way should each knob move?
    opt.step()                      # move them
    if epoch % 40 == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.4f}')

pred_t = model_t(X_test_t).argmax(1)                        # 5 PREDICT
acc_t = (pred_t == y_test_t).float().mean()                 # 6 SCORE
print('\npytorch accuracy:', acc_t.item())


Watch the loss fall. **That loop is Topic 3's training-loop diagram, written out as code.**
`LogisticRegression.fit()` was doing the same thing behind a single method call.

Do not worry about `zero_grad`, `backward`, or what an optimizer is. Those are Topics 6 to 8.


## Part 5. Watch the learning happen

The one thing a notebook can show that a static page cannot: the error falling, epoch by epoch.


In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)
m = nn.Linear(4, 3)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
losses = []
for epoch in range(200):
    opt.zero_grad()
    loss = loss_fn(m(X_train_t), y_train_t)
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('The error falling, one pass at a time')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


That curve *is* training. Every point is one trip around the loop in box 4.


---
## Your turn

1. Change `test_size` from `0.2` to `0.4`. Does accuracy go up or down? Why might that be?
2. Set `max_iter=5` in the scikit-learn model. What happens, and what does the warning tell you?
3. Swap `LogisticRegression` for `DecisionTreeClassifier` from `sklearn.tree`.
   How many of the six boxes did you have to change?
4. In the PyTorch loop, change `lr=0.1` to `lr=0.001` and re-plot. What happened to the curve?


In [ ]:
# your work here
